In [1]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from src.data import load_clean

from src.features import add_distance_features, add_listing_features
df = add_listing_features(add_distance_features(load_clean()))

print(df.shape)  # ожидаем 33769 строк

(33769, 99)


In [2]:
num_cols = ["accommodates", "bedrooms", "beds", "latitude", "longitude",
            "dist_colosseum_km", "dist_termini_km", "dist_st_peters_km",
            "dist_pantheon_km", "dist_trevi_km", "dist_nearest_landmark_km"]
cat_cols = ["room_type", "neighbourhood_cleansed"]

X = df[num_cols + cat_cols].copy()
for c in cat_cols:
    X[c] = X[c].astype("category")
y = df["log_price"]

In [3]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_base = np.zeros(len(df))
oof_lgb = np.zeros(len(df))

for tr, te in kf.split(df):
    train, test = df.iloc[tr], df.iloc[te]

    # бейзлайн: медиана log_price по (район, тип жилья)
    med = train.groupby(["neighbourhood_cleansed", "room_type"])["log_price"].median()
    keys = pd.MultiIndex.from_frame(test[["neighbourhood_cleansed", "room_type"]])
    oof_base[te] = med.reindex(keys).fillna(train["log_price"].median()).values

    # LightGBM
    model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05,
                              num_leaves=31, random_state=42, verbose=-1)
    model.fit(X.iloc[tr], y.iloc[tr])
    oof_lgb[te] = model.predict(X.iloc[te])

In [4]:
is_centre = (df["neighbourhood_cleansed"] == "I Centro Storico").values

def report(name, pred_log):
    err = np.abs(np.expm1(pred_log) - df["price"].values)
    print(f"{name:10s} MAE all: {err.mean():6.1f} | centre: {err[is_centre].mean():6.1f} "
          f"| rest: {err[~is_centre].mean():6.1f} | R2(log): {r2_score(y, pred_log):.3f}")

report("baseline", oof_base)
report("lightgbm", oof_lgb)

baseline   MAE all:   86.7 | centre:  108.6 | rest:   63.0 | R2(log): 0.302
lightgbm   MAE all:   66.0 | centre:   79.1 | rest:   51.7 | R2(log): 0.578


In [5]:
model.fit(X, y)
(pd.Series(model.feature_importances_, index=X.columns)
   .sort_values(ascending=False))

latitude                    2061
longitude                   1564
dist_colosseum_km           1558
dist_st_peters_km           1314
dist_nearest_landmark_km    1301
dist_pantheon_km            1242
dist_termini_km             1195
accommodates                1152
dist_trevi_km               1135
beds                         934
bedrooms                     672
room_type                    533
neighbourhood_cleansed       339
dtype: int32

In [6]:
def mdape(pred_log, mask):
    p, t = np.expm1(pred_log), df["price"].values
    return np.median(np.abs(p - t)[mask] / t[mask]) * 100

print("MdAPE centre:", round(mdape(oof_lgb, is_centre), 1), "%")
print("MdAPE rest:  ", round(mdape(oof_lgb, ~is_centre), 1), "%")

(pd.Series(model.booster_.feature_importance("gain"), index=X.columns)
   .sort_values(ascending=False).round(0))

MdAPE centre: 20.4 %
MdAPE rest:   19.9 %


dist_pantheon_km            23922.0
accommodates                20089.0
bedrooms                    12223.0
dist_trevi_km                6954.0
dist_nearest_landmark_km     1628.0
room_type                    1540.0
latitude                     1511.0
dist_colosseum_km            1464.0
neighbourhood_cleansed       1416.0
longitude                    1291.0
dist_st_peters_km            1280.0
dist_termini_km               960.0
beds                          856.0
dtype: float64

In [7]:
from sklearn.model_selection import GroupKFold

print(df["host_id"].nunique(), "hosts for", len(df), "listings")

oof_grp = np.zeros(len(df))
for tr, te in GroupKFold(n_splits=5).split(df, groups=df["host_id"]):
    m = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05,
                          num_leaves=31, random_state=42, verbose=-1)
    m.fit(X.iloc[tr], y.iloc[tr])
    oof_grp[te] = m.predict(X.iloc[te])

report("lgb-hostgrp", oof_grp)

17974 hosts for 33769 listings
lgb-hostgrp MAE all:   67.5 | centre:   80.8 | rest:   53.0 | R2(log): 0.553


In [8]:
def cv_predict(X, y, groups, make_model=None, n_splits=5):
    oof = np.zeros(len(y))
    for tr, te in GroupKFold(n_splits=n_splits).split(X, y, groups):
        if make_model is None:   # бейзлайн
            train, test = df.iloc[tr], df.iloc[te]
            med = train.groupby(["neighbourhood_cleansed", "room_type"])["log_price"].median()
            keys = pd.MultiIndex.from_frame(test[["neighbourhood_cleansed", "room_type"]])
            oof[te] = med.reindex(keys).fillna(train["log_price"].median()).values
        else:
            m = make_model()
            m.fit(X.iloc[tr], y.iloc[tr])
            oof[te] = m.predict(X.iloc[te])
    return oof

make_lgb = lambda: lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05,
                                     num_leaves=31, random_state=42, verbose=-1)

oof_base_g = cv_predict(X, y, df["host_id"])
report("baseline-g", oof_base_g)

baseline-g MAE all:   86.8 | centre:  108.6 | rest:   63.1 | R2(log): 0.299


In [9]:
num_cols2 = num_cols + ["n_amenities", "bathrooms_num", "minimum_nights"]
cat_cols2 = cat_cols + ["property_type"]

X2 = df[num_cols2 + cat_cols2].copy()
for c in cat_cols2:
    X2[c] = X2[c].astype("category")

def cv_predict(X, y, groups, make_model=None, n_splits=5):
    oof = np.zeros(len(y))
    for tr, te in GroupKFold(n_splits=n_splits).split(X, y, groups):
        if make_model is None:   # бейзлайн
            train, test = df.iloc[tr], df.iloc[te]
            med = train.groupby(["neighbourhood_cleansed", "room_type"])["log_price"].median()
            keys = pd.MultiIndex.from_frame(test[["neighbourhood_cleansed", "room_type"]])
            oof[te] = med.reindex(keys).fillna(train["log_price"].median()).values
        else:
            m = make_model()
            m.fit(X.iloc[tr], y.iloc[tr])
            oof[te] = m.predict(X.iloc[te])
    return oof

make_lgb = lambda: lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05,
                                     num_leaves=31, random_state=42, verbose=-1)

In [10]:
oof_base_g = cv_predict(X, y, df["host_id"])
oof_lgb2 = cv_predict(X2, y, df["host_id"], make_lgb)

report("baseline-g", oof_base_g)
report("lgb-v1-g", oof_grp)      # уже посчитано в ячейке 7
report("lgb-v2-g", oof_lgb2)

baseline-g MAE all:   86.8 | centre:  108.6 | rest:   63.1 | R2(log): 0.299
lgb-v1-g   MAE all:   67.5 | centre:   80.8 | rest:   53.0 | R2(log): 0.553
lgb-v2-g   MAE all:   64.1 | centre:   76.8 | rest:   50.2 | R2(log): 0.600
